In [2]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import requests
from io import StringIO

load_dotenv()

True

In [4]:
engine = create_engine(os.getenv("DB_URL"))

In [5]:
# pattern: use the same DB_URL from .env as your load scripts
df = pd.read_sql("select * from ballon_dor.ballon_dor_features", engine)
df.shape

(408, 14)

In [6]:
# pattern: 1 = real stats, 0 = no stats row
df["has_stats"] = df["goals"].notna().astype(int)
df["has_stats"].value_counts()

has_stats
1    396
0     12
Name: count, dtype: int64

In [8]:
# pattern for ONE column: adapt it for goals, assists, minutes
median_goals = df["goals"].median()      # median ignores NaN by default
df["goals"] = df["goals"].fillna(median_goals)

median_minutes = df["minutes"].median()
df["minutes"] = df["minutes"].fillna(median_minutes)

median_assists = df["assists"].median()
df["assists"] = df["assists"].fillna(median_assists)

df["main_position"] = df["main_position"].fillna("unknown")

In [11]:
df["has_stats"].value_counts()

has_stats
1    396
0     12
Name: count, dtype: int64

In [12]:
df[["goals","assists","minutes","ucl_stage","intl_stage"]].describe()

,goals,assists,minutes,ucl_stage,intl_stage
count,408.000000,408.000000,408.000000,408.000000,408.000000
mean,12.502451,6.220588,2543.857843,3.375000,2.095588
std,10.999441,4.472726,495.433049,1.987445,2.329847
min,0.000000,0.000000,958.000000,0.000000,0.000000
25%,3.000000,3.000000,2196.500000,2.000000,0.000000
50%,10.000000,6.000000,2606.000000,4.000000,1.000000
75%,21.000000,9.000000,2905.500000,5.000000,4.000000
max,50.000000,20.000000,3420.000000,6.000000,6.000000


In [13]:
# pattern: target = 1 for the season's winner, else 0
df["is_winner"] = (df["final_rank"] == 1).astype(int)
df[["goals","assists","minutes","ucl_stage","intl_stage","won_league","is_winner"]].corr()["is_winner"]

goals         0.277968
assists       0.214849
minutes       0.076835
ucl_stage     0.133688
intl_stage    0.075931
won_league    0.044839
is_winner     1.000000
Name: is_winner, dtype: float64

In [15]:
# pattern: season average, aligned to each row
season_avg = df.groupby("season")["goals"].transform("mean")
df["goals_vs_season"] = df["goals"] - season_avg
df


,season,player_name,main_club,final_rank,won_league,ucl_stage,club,minutes,goals,assists,main_position,nation,tournament,intl_stage,has_stats,is_winner,goals_vs_season
0,2010,Andrés Iniesta,Barcelona,2,1,4,Barcelona,1905.0,1.0,3.0,MF,ESP,WC,6,1,0,-9.086957
1,2010,Arjen Robben,Bayern Munich,14,1,5,Bayern Munich,1778.0,16.0,5.0,FW,NED,WC,5,1,0,5.913043
2,2010,Asamoah Gyan,Sunderland,18,0,0,Rennes,2079.0,13.0,2.0,FW,GHA,WC,3,1,0,2.913043
3,2010,Bastian Schweinsteiger,Bayern Munich,16,1,5,Bayern Munich,2903.0,2.0,2.0,DF,GER,WC,4,1,0,-8.086957
4,2010,Carles Puyol,Barcelona,11,1,4,Barcelona,2712.0,1.0,1.0,DF,ESP,WC,6,1,0,-9.086957
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403,2025,Serhou Guirassy,Borussia Dortmund,21,0,3,Dortmund,2598.0,21.0,4.0,FW,GUI,None,0,1,0,9.466667
404,2025,Viktor Gyökeres,Sporting CP,15,0,1,None,2606.0,10.0,6.0,unknown,SWE,None,0,0,0,-1.533333
405,2025,Vinicius Júnior,Real Madrid,16,0,3,Real Madrid,2253.0,11.0,8.0,FW,BRA,None,0,1,0,-0.533333
406,2025,Virgil van Dijk,Liverpool,28,1,2,Liverpool,3330.0,3.0,1.0,DF,NED,None,0,1,0,-8.533333


In [17]:
df = df.drop(columns=["goals_vs_season"])

In [18]:
# pattern: split by season, not by random rows
train = df[df["season"] <= 2022]
test  = df[df["season"] >= 2023]
train.shape, test.shape, train["is_winner"].sum(), test["is_winner"].sum()

((318, 16), (90, 16), np.int64(12), np.int64(3))

In [19]:
# pattern: the row with the most goals in each season
pred = df.loc[df.groupby("season")["goals"].idxmax()]
pred["is_winner"].sum()   # how many of those rows are real winners

np.int64(4)

In [26]:
# pattern: choose the model inputs, then one-hot the position
features = ["goals", "assists", "minutes", "ucl_stage", "intl_stage", "won_league", "has_stats"]
X = pd.get_dummies(df[features + ["main_position"]], columns=["main_position"], drop_first=True)
y = df["is_winner"]
X.shape

(408, 11)

In [55]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression# pattern
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0)

#model = make_pipeline(
   # StandardScaler(), 
  #  LogisticRegression(class_weight="balanced")
 #   )

# pattern: hold out one season
test_mask = df["season"] == 2019
model.fit(X[~test_mask], y[~test_mask])
proba = model.predict_proba(X[test_mask])[:, 1]   # probability of "winner" for each 2019 nominee

In [56]:
season_df = df[test_mask]
best = proba.argmax()

predicted = season_df.iloc[best]["player_name"]
actual = season_df[season_df["final_rank"] == 1]["player_name"].values[0]

print(f"Predicted: {predicted}, Actual: {actual}")

Predicted: Lionel Messi, Actual: Lionel Messi


In [57]:
hits = 0
hits_top3 = 0

for s in sorted(df["season"].unique()):
    test_mask = df["season"] == s
    season_df = df[test_mask]
    model.fit(X[~test_mask], y[~test_mask])
    proba = model.predict_proba(X[test_mask])[:, 1]
    best = proba.argmax()
    predicted = season_df.iloc[best]["player_name"]
    actual = season_df[season_df["final_rank"] == 1]["player_name"].values[0]
    top3 = proba.argsort()[::-1][:3]
    top3_names = season_df.iloc[top3]["player_name"].tolist()
    
    if predicted == actual:
        hits += 1
    if actual in top3_names:
        hits_top3 += 1
    print(s, actual, "->", top3_names)
print(hits, "of", df["season"].nunique())
print(hits_top3, "of", df["season"].nunique(), "in top 3")

2010 Lionel Messi -> ['Lionel Messi', 'Xavi', 'Wesley Sneijder']
2011 Lionel Messi -> ['Lionel Messi', 'Cristiano Ronaldo', 'David Villa']
2012 Lionel Messi -> ['Cristiano Ronaldo', 'Andrés Iniesta', 'Robin van Persie']
2013 Cristiano Ronaldo -> ['Lionel Messi', 'Cristiano Ronaldo', 'Arjen Robben']
2014 Cristiano Ronaldo -> ['Cristiano Ronaldo', 'Yaya Touré', 'Ángel Di María']
2015 Lionel Messi -> ['Cristiano Ronaldo', 'Lionel Messi', 'Neymar']
2016 Cristiano Ronaldo -> ['Cristiano Ronaldo', 'Lionel Messi', 'Zlatan Ibrahimović']
2017 Cristiano Ronaldo -> ['Cristiano Ronaldo', 'Lionel Messi', 'Luis Suárez']
2018 Luka Modrić -> ['Mohamed Salah', 'Cristiano Ronaldo', 'Kevin De Bruyne']
2019 Lionel Messi -> ['Lionel Messi', 'Mohamed Salah', 'Roberto Firmino']
2021 Lionel Messi -> ['Lionel Messi', 'Mason Mount', 'Nicolò Barella']
2022 Karim Benzema -> ['Karim Benzema', 'Mohamed Salah', 'Kylian Mbappé']
2023 Lionel Messi -> ['Erling Haaland', 'Julián Álvarez', 'Lautaro Martínez']
2024 Rodri 

In [45]:
# pattern
model.fit(X, y)
print(list(zip(X.columns, model[-1].coef_[0])))

[('goals', np.float64(1.7757312945746968)), ('assists', np.float64(0.9387777203296608)), ('minutes', np.float64(-0.05509016887281763)), ('ucl_stage', np.float64(1.9304499687727037)), ('intl_stage', np.float64(1.3244440312193684)), ('won_league', np.float64(0.22534094877703104)), ('has_stats', np.float64(0.03434524009537292)), ('main_position_FW', np.float64(0.6905261453238563)), ('main_position_GK', np.float64(-0.11086316110124084)), ('main_position_MF', np.float64(1.033644245595857)), ('main_position_unknown', np.float64(-0.03434524009537297))]


.../miniconda3/envs/quant/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
.../miniconda3/envs/quant/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
.../miniconda3/envs/quant/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
.../miniconda3/envs/quant/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:333: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
.../miniconda3/envs/quant/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:333: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * 